In [9]:
import pandas as pd
import numpy as np
from pathlib import Path


BASE_DIR = Path.cwd().parents[0]
DATA_DIR = BASE_DIR / "datasets"
data_path = DATA_DIR / "final_sample.parquet"
table = pd.read_parquet(data_path)

In [10]:
len(table)

7947995

In [11]:
import geopandas as gpd


BASE_DIR = Path.cwd().parents[0]
DATA_DIR = BASE_DIR / "reference"
shp_path = DATA_DIR / "taxi_zones.shp"
gdf_zones = gpd.read_file(shp_path)

gdf_proj = gdf_zones.to_crs(epsg=2263)
centroids = gdf_proj.geometry.centroid
centroids_wgs84 = gpd.GeoSeries(centroids, crs=gdf_proj.crs).to_crs(epsg=4326)

gdf_zones["lon"] = centroids_wgs84.x
gdf_zones["lat"] = centroids_wgs84.y


zone_info = gdf_zones[["LocationID", "borough", "zone", "lon", "lat"]].copy()

pu_zones = zone_info.rename(columns={
    "LocationID": "PULocationID",
    "borough": "PU_Borough",
    "zone": "PU_Zone",
    "lon": "PU_lon",
    "lat": "PU_lat"
})

do_zones = zone_info.rename(columns={
    "LocationID": "DOLocationID",
    "borough": "DO_Borough",
    "zone": "DO_Zone",
    "lon": "DO_lon",
    "lat": "DO_lat"
})

table = table.merge(pu_zones, on="PULocationID", how="left")
table = table.merge(do_zones, on="DOLocationID", how="left")

table[[
    "PULocationID", "PU_Borough", "PU_Zone", "PU_lon", "PU_lat",
    "DOLocationID", "DO_Borough", "DO_Zone", "DO_lon", "DO_lat"
]].head()

,PULocationID,PU_Borough,PU_Zone,PU_lon,PU_lat,DOLocationID,DO_Borough,DO_Zone,DO_lon,DO_lat
0,262,Manhattan,Yorkville East,-73.946510,40.775932,74,Manhattan,East Harlem North,-73.937346,40.801169
1,229,Manhattan,Sutton Place/Turtle Bay North,-73.965146,40.756729,237,Manhattan,Upper East Side South,-73.965635,40.768615
2,45,Manhattan,Chinatown,-73.998151,40.712459,261,Manhattan,World Trade Center,-74.013023,40.709139
3,237,Manhattan,Upper East Side South,-73.965635,40.768615,141,Manhattan,Lenox Hill West,-73.959635,40.766948
4,229,Manhattan,Sutton Place/Turtle Bay North,-73.965146,40.756729,140,Manhattan,Lenox Hill East,-73.954739,40.765484


In [12]:
table["airport_fee"] = table["airport_fee"].combine_first(table["Airport_fee"])

table = table.drop(columns=["Airport_fee"], errors="ignore")

table["airport_fee"] = table["airport_fee"].fillna(0)

In [13]:
# Денежные столбцы, в которых отрицательные значения могут быть аномалиями.
money_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]

money_cols = [col for col in money_cols if col in table.columns]

for col in money_cols:
    table[col] = pd.to_numeric(table[col], errors="coerce")

# Если improvement_surcharge отрицательный, считаем, что вся строка записана
# со знаком минус по денежным полям, и берём эти поля по модулю.
anomaly_mask = table["improvement_surcharge"] < 0

table.loc[anomaly_mask, money_cols] = table.loc[anomaly_mask, money_cols].abs()

print("Количество строк, исправленных через abs():", anomaly_mask.sum())

# Если improvement_surcharge уже неотрицательный, но в других денежных столбцах
# всё равно есть отрицательные значения, такие строки считаем некорректными и удаляем.
negative_money_mask = table[money_cols].lt(0).any(axis=1)
invalid_negative_mask = (table["improvement_surcharge"] >= 0) & negative_money_mask

print(
    "Количество удалённых строк с необъяснимыми отрицательными денежными значениями:",
    invalid_negative_mask.sum()
)

table = table.loc[~invalid_negative_mask].copy()


Количество строк, исправленных через abs(): 98261
Количество удалённых строк с необъяснимыми отрицательными денежными значениями: 13404


In [14]:

jfk_mask = (
    table["RatecodeID"].isna() &
    (
        (table["PULocationID"] == 132) |
        (table["DOLocationID"] == 132)
    )
)

newark_mask = (
    table["RatecodeID"].isna() &
    (
        (table["PULocationID"] == 1) |
        (table["DOLocationID"] == 1)
    )
)

table.loc[jfk_mask, "RatecodeID"] = 2
table.loc[newark_mask, "RatecodeID"] = 3

table["RatecodeID"] = table["RatecodeID"].fillna(99)

In [15]:
table.loc[
    (table["passenger_count"] <= 0) |
    (table["passenger_count"] > 6),
    "passenger_count"
] = pd.NA

In [16]:
table["pickup_hour"] = table["tpep_pickup_datetime"].dt.hour

table["distance_group"] = pd.cut(
    table["trip_distance"],
    bins=[0, 1, 3, 7, 15, 1000],
    labels=["very_short", "short", "medium", "long", "very_long"]
)

In [17]:
table["store_and_fwd_flag"] = table["store_and_fwd_flag"].fillna("Unknown")

In [18]:
table["distance_group"] = pd.cut(
    table["trip_distance"],
    bins=[-0.01, 1, 3, 7, 15, np.inf],
    labels=["very_short", "short", "medium", "long", "very_long"]
)

table["distance_group"].isna().sum()

np.int64(0)

In [19]:
table.loc[
    (table["passenger_count"] <= 0) |
    (table["passenger_count"] > 6),
    "passenger_count"
] = pd.NA

table["passenger_count"] = table["passenger_count"].fillna(
    table.groupby(
        ["RatecodeID", "payment_type", "pickup_hour", "distance_group"],
        observed=True
    )["passenger_count"].transform("median")
)

table["passenger_count"] = table["passenger_count"].fillna(
    table.groupby(
        ["RatecodeID", "payment_type", "pickup_hour"],
        observed=True
    )["passenger_count"].transform("median")
)

table["passenger_count"] = table["passenger_count"].fillna(
    table.groupby("RatecodeID")["passenger_count"].transform("median")
)

table["passenger_count"] = table["passenger_count"].fillna(1)

table["passenger_count"] = table["passenger_count"].round().astype("int64")

table["passenger_count"].isna().sum()

np.int64(0)

In [20]:
zone_coord_cols = ["PU_lon", "PU_lat", "DO_lon", "DO_lat"]

for col in zone_coord_cols:
    table[col] = table[col].fillna(0)

table[zone_coord_cols].isna().sum()

PU_lon    0
PU_lat    0
DO_lon    0
DO_lat    0
dtype: int64

In [21]:
# Убираем оставшиеся пропуски
table["congestion_surcharge"] = table["congestion_surcharge"].fillna(0)

zone_text_cols = ["PU_Borough", "PU_Zone", "DO_Borough", "DO_Zone"]
table[zone_text_cols] = table[zone_text_cols].fillna("Unknown")

table[["congestion_surcharge", *zone_text_cols]].isna().sum()

congestion_surcharge    0
PU_Borough              0
PU_Zone                 0
DO_Borough              0
DO_Zone                 0
dtype: int64

In [22]:
technical_cols = [
    "source_file",
    "sample_year",
    "sample_month",
    "month_total_rows",
    "month_sample_rows",
    "target_sample_frac",
    "actual_sample_frac",
    "cap_was_applied",
]
table = table.drop(columns=technical_cols)

In [23]:
table.isna().sum()

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
airport_fee              0
PU_Borough               0
PU_Zone                  0
PU_lon                   0
PU_lat                   0
DO_Borough               0
DO_Zone                  0
DO_lon                   0
DO_lat                   0
pickup_hour              0
distance_group           0
dtype: int64

In [24]:
for i in table.columns:
    print(i, table[i].isna().sum())

VendorID 0
tpep_pickup_datetime 0
tpep_dropoff_datetime 0
passenger_count 0
trip_distance 0
RatecodeID 0
store_and_fwd_flag 0
PULocationID 0
DOLocationID 0
payment_type 0
fare_amount 0
extra 0
mta_tax 0
tip_amount 0
tolls_amount 0
improvement_surcharge 0
total_amount 0
congestion_surcharge 0
airport_fee 0
PU_Borough 0
PU_Zone 0
PU_lon 0
PU_lat 0
DO_Borough 0
DO_Zone 0
DO_lon 0
DO_lat 0
pickup_hour 0
distance_group 0


In [25]:
table["airport_fee"] = table["airport_fee"].fillna(0)
table["passenger_count"] = table["passenger_count"].fillna(
    table["passenger_count"].median()
)
table["RatecodeID"] = table["RatecodeID"].fillna(99)
table["pickup_hour"] = table["tpep_pickup_datetime"].dt.hour
table["passenger_count"] = table["passenger_count"].fillna(
    table.groupby(["RatecodeID", "payment_type", "pickup_hour"])["passenger_count"]
    .transform("median")
)

table["passenger_count"] = table["passenger_count"].fillna(
    table.groupby("RatecodeID")["passenger_count"].transform("median")
)

table["passenger_count"] = table["passenger_count"].fillna(
    table["passenger_count"].median()
)

In [26]:
table["duration_min"] = (
    table["tpep_dropoff_datetime"] - table["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [27]:
from IPython.display import display

base_money_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
]

optional_total_cols = [
    "airport_fee",
    "congestion_surcharge",
]

base_money_cols = [
    col for col in base_money_cols
    if col in table.columns
]

optional_total_cols = [
    col for col in optional_total_cols
    if col in table.columns
]

money_cols = base_money_cols + optional_total_cols

for col in set(money_cols + ["total_amount"]):
    table[col] = pd.to_numeric(table[col], errors="coerce")

table["calculated_total_full"] = (
    table[money_cols]
    .fillna(0)
    .sum(axis=1)
)

airport_fee_values = (
    table["airport_fee"].fillna(0)
    if "airport_fee" in table.columns
    else pd.Series(0, index=table.index)
)

congestion_surcharge_values = (
    table["congestion_surcharge"].fillna(0)
    if "congestion_surcharge" in table.columns
    else pd.Series(0, index=table.index)
)

table["total_difference"] = (
    table["calculated_total_full"] - table["total_amount"]
)

abs_difference = table["total_difference"].abs()

valid_total_mask = pd.Series(False, index=table.index)

# 1. Полное совпадение.
valid_total_mask |= np.isclose(
    abs_difference,
    0,
    atol=0.05,
)

# 2. Расхождение равно airport_fee из строки.
valid_total_mask |= np.isclose(
    abs_difference,
    airport_fee_values.abs(),
    atol=0.05,
)

# 3. Расхождение равно congestion_surcharge из строки.
valid_total_mask |= np.isclose(
    abs_difference,
    congestion_surcharge_values.abs(),
    atol=0.05,
)

# 4. Расхождение равно airport_fee + congestion_surcharge из строки.
valid_total_mask |= np.isclose(
    abs_difference,
    (airport_fee_values + congestion_surcharge_values).abs(),
    atol=0.05,
)

# 5. Явно разрешённые фиксированные расхождения.
# Это важно для случаев, когда в колонке сбор стоит 0/NaN,
# но расхождение всё равно равно типичному размеру сбора.
allowed_fixed_differences = [
    1.25,
    2.50,
    3.75,
]

for allowed_diff in allowed_fixed_differences:
    valid_total_mask |= np.isclose(
        abs_difference,
        allowed_diff,
        atol=0.05,
    )

bad_total_rows = table.loc[~valid_total_mask].copy()

print("Количество удалённых строк с некорректной итоговой суммой:", len(bad_total_rows))

debug_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "airport_fee",
    "congestion_surcharge",
    "total_amount",
    "calculated_total_full",
    "total_difference",
]

debug_cols = [      
    col for col in debug_cols
    if col in bad_total_rows.columns
]

# display(
#     bad_total_rows[debug_cols]
#     .head(1000)
#     .round(2)
# )

table = table.loc[valid_total_mask].copy()

Количество удалённых строк с некорректной итоговой суммой: 4349


In [28]:
max_duration_min = 300
max_trip_distance_miles = 150
max_avg_speed_mph = 80
max_total_amount = 500
max_fare_amount = 500
max_tolls_amount = 200

coord_cols = ["PU_lon", "PU_lat", "DO_lon", "DO_lat"]
money_check_cols = list(dict.fromkeys(money_cols + ["total_amount"]))
avg_speed_mph = table["trip_distance"] / (table["duration_min"] / 60)

table_clean = table[
    table["duration_min"].between(1, max_duration_min) &
    table["trip_distance"].between(0, max_trip_distance_miles) &
    (avg_speed_mph <= max_avg_speed_mph) &
    table["total_amount"].between(0, max_total_amount) &
    table["fare_amount"].between(0, max_fare_amount) &
    table["tolls_amount"].between(0, max_tolls_amount) &
    table[money_check_cols].ge(0).all(axis=1) &
    (table["passenger_count"] > 0) &
    ~((table["trip_distance"] == 0) & (table["fare_amount"] > 5)) &
    ~((table["trip_distance"] > 0.1) & (table["fare_amount"] == 0)) &
    table[coord_cols].notna().all(axis=1) &
    (table[coord_cols] != 0).all(axis=1)
].copy()

table = table[~((table["trip_distance"] == 0) & (table["total_amount"] > 0))]

table_clean = table_clean.drop(
    columns=["calculated_total_full", "total_difference"],
    errors="ignore"
)

In [29]:
table_clean["tpep_pickup_datetime"] = pd.to_datetime(table_clean["tpep_pickup_datetime"])
table_clean["tpep_dropoff_datetime"] = pd.to_datetime(table_clean["tpep_dropoff_datetime"])

table_clean = table_clean[
    (table_clean["tpep_pickup_datetime"] >= "2023-01-01") &
    (table_clean["tpep_dropoff_datetime"] < "2025-01-01")
].copy()

In [30]:
money_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee"
]

calculated_amount = table_clean[money_cols].sum(axis=1)
insert_pos = table_clean.columns.get_loc("total_amount") + 1
table_clean.insert(insert_pos, "calculated_amount", calculated_amount)

In [32]:
BASE_DIR = Path.cwd().parents[0]
DATA_DIR = BASE_DIR / "datasets"
OUTPUT_PATH = DATA_DIR / "my_clean_3_with_weather.parquet"

table_clean.to_parquet(OUTPUT_PATH, index=False, compression="zstd")

print("Saved:", OUTPUT_PATH)
print("Rows:", len(table_clean))
print("Columns:", table_clean.shape[1])
print("Missing values:", int(table_clean.isna().sum().sum()))

Saved: C:\Users\user\Documents\Project\datasets\short_my_clean_3_with_weather.parquet
Rows: 7655312
Columns: 31
Missing values: 0


In [33]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
table_clean

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,calculated_amount,congestion_surcharge,airport_fee,PU_Borough,PU_Zone,PU_lon,PU_lat,DO_Borough,DO_Zone,DO_lon,DO_lat,pickup_hour,distance_group,duration_min
0,2,2023-01-29 17:52:02,2023-01-29 17:56:43,1,1.17,1.0,N,262,74,2,7.2,0.0,0.5,0.00,0.0,1.0,11.20,11.20,2.5,0.00,Manhattan,Yorkville East,-73.946510,40.775932,Manhattan,East Harlem North,-73.937346,40.801169,17,short,4.683333
1,1,2023-01-08 15:57:24,2023-01-08 16:02:47,1,0.90,1.0,N,229,237,2,6.5,2.5,0.5,0.00,0.0,1.0,10.50,13.00,2.5,0.00,Manhattan,Sutton Place/Turtle Bay North,-73.965146,40.756729,Manhattan,Upper East Side South,-73.965635,40.768615,15,very_short,5.383333
2,2,2023-01-21 19:38:01,2023-01-21 19:45:02,1,0.95,1.0,N,45,261,1,7.9,0.0,0.5,2.38,0.0,1.0,14.28,14.28,2.5,0.00,Manhattan,Chinatown,-73.998151,40.712459,Manhattan,World Trade Center,-74.013023,40.709139,19,very_short,7.016667
3,2,2023-01-23 16:07:31,2023-01-23 16:26:46,5,0.88,1.0,N,237,141,1,16.3,2.5,0.5,1.50,0.0,1.0,24.30,24.30,2.5,0.00,Manhattan,Upper East Side South,-73.965635,40.768615,Manhattan,Lenox Hill West,-73.959635,40.766948,16,very_short,19.250000
4,2,2023-01-26 21:21:08,2023-01-26 21:24:48,2,1.03,1.0,N,229,140,1,6.5,1.0,0.5,2.30,0.0,1.0,13.80,13.80,2.5,0.00,Manhattan,Sutton Place/Turtle Bay North,-73.965146,40.756729,Manhattan,Lenox Hill East,-73.954739,40.765484,21,short,3.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7947989,1,2024-12-22 19:13:20,2024-12-22 19:28:09,1,2.70,1.0,N,141,74,1,14.2,2.5,0.5,3.65,0.0,1.0,21.85,24.35,2.5,0.00,Manhattan,Lenox Hill West,-73.959635,40.766948,Manhattan,East Harlem North,-73.937346,40.801169,19,short,14.816667
7947990,2,2024-12-04 15:08:48,2024-12-04 15:42:17,2,11.04,1.0,N,132,138,1,45.0,5.0,0.5,10.30,0.0,1.0,63.55,63.55,0.0,1.75,Queens,JFK Airport,-73.786530,40.646985,Queens,LaGuardia Airport,-73.873628,40.774376,15,long,33.483333
7947991,2,2024-12-16 00:11:09,2024-12-16 00:17:22,1,0.85,1.0,N,144,4,1,7.9,1.0,0.5,1.94,0.0,1.0,14.84,14.84,2.5,0.00,Manhattan,Little Italy/NoLiTa,-73.996919,40.720889,Manhattan,Alphabet City,-73.976968,40.723752,0,very_short,6.216667
7947992,2,2024-12-05 07:38:08,2024-12-05 07:49:33,1,1.07,1.0,N,162,230,1,11.4,0.0,0.5,4.62,0.0,1.0,20.02,20.02,2.5,0.00,Manhattan,Midtown East,-73.972356,40.756688,Manhattan,Times Sq/Theatre District,-73.984197,40.759818,7,short,11.416667
